In [3]:
!pip install numpy==1.26.4 -q
!pip install scikit-surprise -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 64.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you hav

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
from surprise import Dataset, Reader, NormalPredictor
from surprise.model_selection import cross_validate

# Re-load datasets after restart
def load_ratebeer(path):
    records = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = eval(line)
                num, denom = record['review/overall'].split('/')
                records.append({
                    'user_id': record['review/profileName'],
                    'item_id': record['beer/beerId'],
                    'rating': (float(num) / float(denom)) * 5
                })
            except:
                continue
    return pd.DataFrame(records)

def load_beeradvocate(path):
    records = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = eval(line)
                records.append({
                    'user_id': record['review/profileName'],
                    'item_id': record['beer/beerId'],
                    'rating': float(record['review/overall'])
                })
            except:
                continue
    return pd.DataFrame(records)

rb_df = load_ratebeer('/content/drive/MyDrive/CMPE256/ratebeer.json')
ba_df = load_beeradvocate('/content/drive/MyDrive/CMPE256/beeradvocate.json')
print("Loaded!", rb_df.shape, ba_df.shape)

Loaded! (2924163, 3) (1586614, 3)


In [4]:
from surprise import Dataset, Reader, NormalPredictor
from surprise.model_selection import cross_validate

def run_global_mean(df, dataset_name):
    print(f"\n--- {dataset_name} ---")
    sample = df.sample(n=200000, random_state=42)
    reader = Reader(rating_scale=(0, 5))
    data = Dataset.load_from_df(sample[['user_id', 'item_id', 'rating']], reader)
    model = NormalPredictor()
    results = cross_validate(model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
    print(f"Avg RMSE: {results['test_rmse'].mean():.4f}")
    print(f"Avg MAE:  {results['test_mae'].mean():.4f}")

run_global_mean(rb_df, "RateBeer")
run_global_mean(ba_df, "BeerAdvocate")


--- RateBeer ---
Evaluating RMSE, MAE of algorithm NormalPredictor on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    1.1806  1.1715  1.1745  1.1769  1.1742  1.1755  0.0030  
MAE (testset)     0.9292  0.9247  0.9264  0.9257  0.9264  0.9265  0.0015  
Fit time          0.31    0.44    0.27    0.27    0.27    0.31    0.07    
Test time         0.25    0.46    0.15    0.30    0.16    0.26    0.11    
Avg RMSE: 1.1755
Avg MAE:  0.9265

--- BeerAdvocate ---
Evaluating RMSE, MAE of algorithm NormalPredictor on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9930  0.9930  0.9899  0.9921  0.9847  0.9905  0.0031  
MAE (testset)     0.7831  0.7800  0.7809  0.7797  0.7766  0.7801  0.0021  
Fit time          0.19    0.27    0.29    0.28    0.28    0.26    0.03    
Test time         0.30    0.17    0.31    0.16    0.30    0.25    0.07    
Avg RMSE: 0.9905
Avg MAE:  0.7801


In [5]:
from surprise import BaselineOnly

def run_baseline_only(df, dataset_name):
    print(f"\n--- {dataset_name} ---")
    sample = df.sample(n=200000, random_state=42)
    reader = Reader(rating_scale=(0, 5))
    data = Dataset.load_from_df(sample[['user_id', 'item_id', 'rating']], reader)
    model = BaselineOnly()
    results = cross_validate(model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
    print(f"Avg RMSE: {results['test_rmse'].mean():.4f}")
    print(f"Avg MAE:  {results['test_mae'].mean():.4f}")

run_baseline_only(rb_df, "RateBeer")
run_baseline_only(ba_df, "BeerAdvocate")


--- RateBeer ---
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Evaluating RMSE, MAE of algorithm BaselineOnly on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.6422  0.6426  0.6416  0.6396  0.6410  0.6414  0.0011  
MAE (testset)     0.4688  0.4701  0.4723  0.4685  0.4704  0.4700  0.0014  
Fit time          1.22    1.42    1.29    1.19    1.21    1.27    0.08    
Test time         0.42    0.43    0.28    0.29    0.28    0.34    0.07    
Avg RMSE: 0.6414
Avg MAE:  0.4700

--- BeerAdvocate ---
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Evaluating RMSE, MAE of algorithm BaselineOnly on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.6359  0.6308  0.6306

In [6]:
import pandas as pd

train = pd.read_parquet('/content/drive/MyDrive/SP2026_CMPE256_Group12_Data/basic_train_temporal.parquet')
test = pd.read_parquet('/content/drive/MyDrive/SP2026_CMPE256_Group12_Data/basic_test_temporal.parquet')

print(train.shape, test.shape)
print(train.columns.tolist())
print(train.head())

(3541459, 3) (885365, 3)
['user_id', 'item_id', 'rating']
        user_id item_id  rating
0      ba_Jason  ba_165    3.00
1      ba_Jason  ba_441    4.00
2  rb_lazarus99  rb_121    5.00
3      rb_billb  rb_132    4.25
4       ba_Todd  ba_563    4.00


In [7]:
from surprise import Dataset, Reader, NormalPredictor, BaselineOnly
from surprise.model_selection import cross_validate

reader = Reader(rating_scale=(0, 5))

# Use train set only for CV (as intended)
data = Dataset.load_from_df(train[['user_id', 'item_id', 'rating']], reader)

print("\n--- NormalPredictor (Pipeline data) ---")
results_np = cross_validate(NormalPredictor(), data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"Avg RMSE: {results_np['test_rmse'].mean():.4f}")
print(f"Avg MAE:  {results_np['test_mae'].mean():.4f}")

print("\n--- BaselineOnly (Pipeline data) ---")
results_bo = cross_validate(BaselineOnly(), data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"Avg RMSE: {results_bo['test_rmse'].mean():.4f}")
print(f"Avg MAE:  {results_bo['test_mae'].mean():.4f}")


--- NormalPredictor (Pipeline data) ---
Evaluating RMSE, MAE of algorithm NormalPredictor on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    1.1786  1.1772  1.1790  1.1787  1.1793  1.1786  0.0008  
MAE (testset)     0.9302  0.9294  0.9315  0.9305  0.9312  0.9305  0.0007  
Fit time          9.97    5.74    5.99    6.40    7.34    7.09    1.54    
Test time         7.80    8.96    11.35   9.19    8.67    9.19    1.17    
Avg RMSE: 1.1786
Avg MAE:  0.9305

--- BaselineOnly (Pipeline data) ---
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Evaluating RMSE, MAE of algorithm BaselineOnly on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.5883  0.5882  0.5873  0.5879  0.5876  0.5879  0.0004  
MAE (testset)     0.4306  0.4302  0.4300  0.4307  0.4305  0.4304  0.0003  
Fit

In [8]:
adv_train = pd.read_parquet('/content/drive/MyDrive/SP2026_CMPE256_Group12_Data/advanced_train_temporal.parquet')
adv_test = pd.read_parquet('/content/drive/MyDrive/SP2026_CMPE256_Group12_Data/advanced_test_temporal.parquet')

print(adv_train.shape, adv_test.shape)
print(adv_train.columns.tolist())
print(adv_train.head())

(3541459, 13) (885365, 13)
['user_id', 'item_id', 'rating', 'beer/name', 'beer/brewerId', 'beer/ABV', 'beer/style', 'review/appearance', 'review/aroma', 'review/palate', 'review/taste', 'review/time', 'review/text']
        user_id item_id  rating                beer/name beer/brewerId  \
0      ba_Jason  ba_165    3.00           Pale Rider Ale            56   
1      ba_Jason  ba_441    4.00  Red Feather Premium Ale           166   
2  rb_lazarus99  rb_121    5.00    Stone City Hefeweizen            23   
3      rb_billb  rb_132    4.25      Newcastle Brown Ale           751   
4       ba_Todd  ba_563    4.00                BMF Stout            14   

  beer/ABV                beer/style  review/appearance  review/aroma  \
0            American Pale Ale (APA)                3.5           3.5   
1           American Amber / Red Ale                4.0           3.5   
2        -         German Hefeweizen                5.0           4.0   
3      4.7                 Brown Ale           

In [9]:
from surprise import Dataset, Reader, NormalPredictor, BaselineOnly
from surprise.model_selection import cross_validate

reader = Reader(rating_scale=(0, 5))
data_adv = Dataset.load_from_df(adv_train[['user_id', 'item_id', 'rating']], reader)

print("\n--- NormalPredictor (Advanced Pipeline) ---")
results_np = cross_validate(NormalPredictor(), data_adv, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"Avg RMSE: {results_np['test_rmse'].mean():.4f}")
print(f"Avg MAE:  {results_np['test_mae'].mean():.4f}")

print("\n--- BaselineOnly (Advanced Pipeline) ---")
results_bo = cross_validate(BaselineOnly(), data_adv, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"Avg RMSE: {results_bo['test_rmse'].mean():.4f}")
print(f"Avg MAE:  {results_bo['test_mae'].mean():.4f}")


--- NormalPredictor (Advanced Pipeline) ---
Evaluating RMSE, MAE of algorithm NormalPredictor on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    1.1790  1.1793  1.1792  1.1781  1.1782  1.1787  0.0005  
MAE (testset)     0.9314  0.9309  0.9304  0.9302  0.9305  0.9307  0.0004  
Fit time          4.52    6.47    7.39    7.46    6.20    6.41    1.07    
Test time         9.87    9.81    8.99    8.46    9.86    9.40    0.58    
Avg RMSE: 1.1787
Avg MAE:  0.9307

--- BaselineOnly (Advanced Pipeline) ---
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Evaluating RMSE, MAE of algorithm BaselineOnly on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.5887  0.5873  0.5872  0.5875  0.5886  0.5878  0.0007  
MAE (testset)     0.4309  0.4302  0.4301  0.4299  0.4309  0.4304  0.00

In [10]:
from surprise import Dataset, Reader, BaselineOnly
from collections import defaultdict
import numpy as np

reader = Reader(rating_scale=(0, 5))
data = Dataset.load_from_df(train[['user_id', 'item_id', 'rating']], reader)

# Train on full trainset
trainset = data.build_full_trainset()
model = BaselineOnly()
model.fit(trainset)

print("Model trained!")
print(f"Users: {trainset.n_users}, Items: {trainset.n_items}")

Estimating biases using als...
Model trained!
Users: 51019, Items: 143790


In [11]:
def get_top_n(model, trainset, test_df, n=10):
    # Get all unique items from training
    all_items = [trainset.to_raw_iid(i) for i in trainset.all_items()]

    # Group test by user
    test_user_items = test_df.groupby('user_id')['item_id'].apply(set).to_dict()
    test_user_ratings = test_df.groupby('user_id').apply(
        lambda x: dict(zip(x['item_id'], x['rating']))).to_dict()

    # Get users that exist in both train and test
    train_users = set(trainset.to_raw_uid(u) for u in trainset.all_users())
    eval_users = list(set(test_user_items.keys()) & train_users)[:1000]  # sample 1000 users for speed

    precisions, recalls, ndcgs = [], [], []

    for user in eval_users:
        # Predict scores for all items not seen in training
        predictions = [(item, model.predict(user, item).est) for item in all_items]
        predictions.sort(key=lambda x: x[1], reverse=True)
        top_n_items = [item for item, _ in predictions[:n]]

        # Relevant items = items rated >= 4 in test
        relevant = {item for item, rating in test_user_ratings[user].items() if rating >= 4.0}
        if not relevant:
            continue

        hits = [1 if item in relevant else 0 for item in top_n_items]

        # Precision@10
        precisions.append(sum(hits) / n)

        # Recall@10
        recalls.append(sum(hits) / len(relevant))

        # NDCG@10
        dcg = sum(hit / np.log2(i + 2) for i, hit in enumerate(hits))
        ideal = sum(1 / np.log2(i + 2) for i in range(min(len(relevant), n)))
        ndcgs.append(dcg / ideal if ideal > 0 else 0)

    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

print("Evaluating BaselineOnly...")
p, r, n = get_top_n(model, trainset, test, n=10)
print(f"Precision@10: {p:.4f}")
print(f"Recall@10:    {r:.4f}")
print(f"NDCG@10:      {n:.4f}")

Evaluating BaselineOnly...


/tmp/ipykernel_22211/3764815038.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_user_ratings = test_df.groupby('user_id').apply(


Precision@10: 0.0059
Recall@10:    0.0045
NDCG@10:      0.0063


In [12]:
print("Evaluating BaselineOnly @ k=5...")
p5, r5, n5 = get_top_n(model, trainset, test, n=5)
print(f"Precision@5: {p5:.4f}")
print(f"Recall@5:    {r5:.4f}")
print(f"NDCG@5:      {n5:.4f}")

Evaluating BaselineOnly @ k=5...


/tmp/ipykernel_22211/3764815038.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_user_ratings = test_df.groupby('user_id').apply(


Precision@5: 0.0035
Recall@5:    0.0012
NDCG@5:      0.0038


In [13]:
from surprise import NormalPredictor

# Train NormalPredictor
model_np = NormalPredictor()
model_np.fit(trainset)

print("Evaluating NormalPredictor...")
p, r, n = get_top_n(model_np, trainset, test, n=10)
print(f"Precision@10: {p:.4f}")
print(f"Recall@10:    {r:.4f}")
print(f"NDCG@10:      {n:.4f}")

Evaluating NormalPredictor...


/tmp/ipykernel_22211/3764815038.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_user_ratings = test_df.groupby('user_id').apply(


Precision@10: 0.0007
Recall@10:    0.0013
NDCG@10:      0.0012
